# 04 — Noise, significance, and how long you must wait

*Weeks 5–6. This is the heart of the project.*

This notebook distinguishes two quantities: **expected significance**, which
defines the smooth headline sensitivity curve, and **observed significance**,
which fluctuates because a simulated measurement is a Poisson random draw.

The notebook rebuilds every input from the shared functions, so it runs from a
fresh kernel without notebook 03.


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive.*')
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')

ready


## Poisson counting

If the expected number of events is $N$, the typical statistical spread is
about $\sqrt{N}$. Use the fixed reference seed while developing so that your
noisy figures are reproducible.


In [2]:
rng = np.random.default_rng(cfg.REFERENCE_SEED)
for mean in [10, 100, 1000, 10000]:
    draws = rng.poisson(mean, size=5000)
    print(f'expect {mean:6d}   actual spread = {draws.std():8.1f}   '
          f'(sqrt of mean = {np.sqrt(mean):8.1f})   relative = {draws.std()/mean:6.2%}')

expect     10   actual spread =      3.2   (sqrt of mean =      3.2)   relative = 32.15%
expect    100   actual spread =      9.9   (sqrt of mean =     10.0)   relative =  9.95%
expect   1000   actual spread =     31.7   (sqrt of mean =     31.6)   relative =  3.17%
expect  10000   actual spread =    100.9   (sqrt of mean =    100.0)   relative =  1.01%


## Build the baseline expectation maps

This notebook deliberately rebuilds its inputs from the shared functions. You
can run it from a fresh kernel without running notebook 03 first.


In [3]:
target = mg.Pyramid(
    base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
    density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
    void_radius=cfg.VOID_RADIUS_M, has_void=True,
)
target0 = target.without_void()
detector = np.array(cfg.DETECTOR_POSITION_M, dtype=float)

# Trace the geometry once per target, then scale the exposure exactly.
opacity, ax_deg, ay_deg = sa.opacity_map(target, detector)
opacity0, _, _ = sa.opacity_map(target0, detector)
expected_1d = sa.counts_from_opacity(opacity, ax_deg, ay_deg, 1.0)
expected_1d_0 = sa.counts_from_opacity(opacity0, ax_deg, ay_deg, 1.0)
expected_30d = expected_1d * 30.0
expected_30d_0 = expected_1d_0 * 30.0

signal_idx = sa.signal_pixel(expected_1d, expected_1d_0)
print('signal pixel (iy, ix):', signal_idx)
print('direction (ax, ay)   :', ax_deg[signal_idx[1]], ay_deg[signal_idx[0]], 'deg')
print('1-day expected counts with/without:',
      expected_1d[signal_idx], expected_1d_0[signal_idx])
print('30-day expected significance at fixed pixel:',
      mg.expected_significance(expected_30d, expected_30d_0)[signal_idx])

signal pixel (iy, ix): (np.int64(35), np.int64(20))
direction (ax, ay)   : 0.0 30.0 deg
1-day expected counts with/without: 12.226451150852462 8.284405297092489
30-day expected significance at fixed pixel: 7.501560741705843


### YOUR TURN — make the images noisy

Use `mg.poisson_counts` with the reference seed to make 1-day and 30-day
observed count maps. Plot each observed map divided by the corresponding
**expected no-void** map.

The ratio of a noisy observation to the no-void expectation should fluctuate
around 1.0 away from the chamber. The chamber is the only place where the
average ratio rises above 1.0 systematically.


In [4]:
rng = np.random.default_rng(cfg.REFERENCE_SEED)
observed_1d = mg.poisson_counts(expected_1d, rng)
observed_30d = mg.poisson_counts(expected_30d, rng)

ratio_1d = observed_1d / np.maximum(expected_1d_0, 1e-12)
ratio_30d = observed_30d / np.maximum(expected_30d_0, 1e-12)

extent = [ax_deg[0], ax_deg[-1], ay_deg[0], ay_deg[-1]]

print('1-day observed/expected at signal pixel :', f'{ratio_1d[signal_idx]:.2f}')
print('30-day observed/expected at signal pixel:', f'{ratio_30d[signal_idx]:.2f}')
print('1-day  max/min ratio                    :',
      f'{ratio_1d.max():.2f} / {ratio_1d.min():.2f}')
print('30-day max/min ratio                    :',
      f'{ratio_30d.max():.2f} / {ratio_30d.min():.2f}')

1-day observed/expected at signal pixel : 1.45
30-day observed/expected at signal pixel: 1.49
1-day  max/min ratio                    : 2.31 / 0.26
30-day max/min ratio                    : 1.49 / 0.82


## Observed significance

`mg.significance(observed, expected_no_void)` compares one noisy observation
with the no-chamber expectation. It fluctuates from run to run. For this
educational project, 5σ is a predefined headline threshold; it is not a
complete discovery test for a real experiment.

### YOUR TURN

Make significance maps for the 1-day and 30-day observations. State the seed in
the figure caption or log. Do not define the official exposure time from the
largest noisy pixel.


In [5]:
S1_obs = mg.significance(observed_1d, expected_1d_0)
S30_obs = mg.significance(observed_30d, expected_30d_0)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, data, title in [
    (axes[0, 0], ratio_1d, 'Observed / expected-no-void, 1 day'),
    (axes[0, 1], ratio_30d, 'Observed / expected-no-void, 30 days'),
]:
    im = ax.imshow(data, origin='lower', extent=extent, aspect='auto',
                   cmap='inferno', vmin=0.55, vmax=1.65)
    ax.set_title(title)
    ax.set_xlabel('ax (degrees)')
    ax.set_ylabel('ay (degrees)')
    fig.colorbar(im, ax=ax, label='count ratio')

for ax, data, title in [
    (axes[1, 0], S1_obs, 'Observed significance, 1 day'),
    (axes[1, 1], S30_obs, 'Observed significance, 30 days'),
]:
    im = ax.imshow(data, origin='lower', extent=extent, aspect='auto',
                   cmap='RdBu_r', vmin=-5, vmax=12)
    ax.set_title(title)
    ax.set_xlabel('ax (degrees)')
    ax.set_ylabel('ay (degrees)')
    fig.colorbar(im, ax=ax, label=r'observed significance ($\sigma$)')

plt.tight_layout()
plt.savefig('../figures/10_noise_significance.png', dpi=300, bbox_inches='tight')
plt.show()

print('1-day  max observed significance (exploratory):',
      f'{S1_obs.max():.2f} at {np.unravel_index(np.argmax(S1_obs), S1_obs.shape)}')
print('30-day max observed significance (exploratory):',
      f'{S30_obs.max():.2f} at {np.unravel_index(np.argmax(S30_obs), S30_obs.shape)}')
print('observed significance at the fixed signal pixel:',
      f'1 day {S1_obs[signal_idx]:.2f}, 30 days {S30_obs[signal_idx]:.2f}')
print('expected significance at the fixed signal pixel:',
      f'1 day {mg.expected_significance(expected_1d, expected_1d_0)[signal_idx]:.2f}, '
      f'30 days {mg.expected_significance(expected_30d, expected_30d_0)[signal_idx]:.2f}')
print('reference seed:', cfg.REFERENCE_SEED)

1-day  max observed significance (exploratory): 4.07 at (np.int64(38), np.int64(33))
30-day max observed significance (exploratory): 7.70 at (np.int64(35), np.int64(20))
observed significance at the fixed signal pixel: 1 day 1.29, 30 days 7.70
expected significance at the fixed signal pixel: 1 day 1.37, 30 days 7.50
reference seed: 0


## The main result: expected significance

The official headline curve uses **expected significance**:

`S_expected = (expected_with_void - expected_without_void) / sqrt(expected_without_void)`

This is deterministic for fixed parameters and follows the expected `√T`
scaling. It is the correct quantity for a reproducible exposure-time estimate in
this simplified project.

The signal pixel is chosen **once**, from the noiseless expectation maps, and
then held fixed at every exposure time. Choosing a new brightest pixel in each
noisy map would introduce a selection bias.


In [6]:
days_list = np.array([1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 120, 180], dtype=float)

# Because counts are exactly proportional to time, S grows as sqrt(T).
S1_expected = mg.expected_significance(expected_1d, expected_1d_0)[signal_idx]
S_expected = S1_expected * np.sqrt(days_list)

t3 = sa.first_crossing(days_list, S_expected, 3.0)
t5 = sa.first_crossing(days_list, S_expected, 5.0)
t5_exact = 1.0 * (5.0 / S1_expected) ** 2

print('expected S at 1 day:', f'{S1_expected:.4f}')
for d, s in zip(days_list, S_expected):
    print(f'{d:6.0f} days  expected S = {s:6.3f}')
print(f'interpolated 3-sigma crossing: {t3:.2f} days')
print(f'interpolated 5-sigma crossing: {t5:.2f} days')
print(f'exact sqrt-T 5-sigma crossing: {t5_exact:.2f} days')

# A small ensemble of observed measurements at selected exposures, for the plot.
rng = np.random.default_rng(cfg.REFERENCE_SEED)
obs_days = np.array([1.0, 3.0, 7.0, 14.0, 30.0, 90.0])
obs_mean, obs_std = [], []
for d in obs_days:
    draws = []
    for _ in range(200):
        obs = rng.poisson(expected_1d * d)
        draws.append(mg.significance(obs, expected_1d_0 * d)[signal_idx])
    obs_mean.append(np.mean(draws))
    obs_std.append(np.std(draws))
obs_mean = np.array(obs_mean)
obs_std = np.array(obs_std)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(days_list, S_expected, '-', color='C0',
        label=r'expected significance (smooth, $\propto\sqrt{T}$)')
ax.errorbar(obs_days, obs_mean, yerr=obs_std, fmt='o', color='C3', ms=5,
            capsize=3, label='observed significance (200 Poisson runs, mean ± s.d.)')
ax.axhline(3.0, color='C1', ls='--', lw=1, label=r'3$\sigma$')
ax.axhline(5.0, color='C2', ls='--', lw=1, label=r'5$\sigma$')
ax.axvline(t5, color='C2', ls=':', lw=1)
ax.annotate(f'{t5:.1f} d', xy=(t5, 5.0), xytext=(t5*1.15, 6.5),
            arrowprops=dict(arrowstyle='->', lw=0.8), color='C2')
ax.set_xscale('log')
ax.set_xlabel('exposure time (days)')
ax.set_ylabel(r'expected significance $S$ ($\sigma$)')
ax.set_title('Exposure curve for the baseline 0.25 m² detector')
ax.grid(alpha=0.3, which='both')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig('../figures/11_exposure_curve.png', dpi=300, bbox_inches='tight')
plt.show()

expected S at 1 day: 1.3696
     1 days  expected S =  1.370
     2 days  expected S =  1.937
     3 days  expected S =  2.372
     5 days  expected S =  3.062
     7 days  expected S =  3.624
    10 days  expected S =  4.331
    14 days  expected S =  5.125
    21 days  expected S =  6.276
    30 days  expected S =  7.502
    45 days  expected S =  9.187
    60 days  expected S = 10.609
    90 days  expected S = 12.993
   120 days  expected S = 15.003
   180 days  expected S = 18.375
interpolated 3-sigma crossing: 4.82 days
interpolated 5-sigma crossing: 13.37 days
exact sqrt-T 5-sigma crossing: 13.33 days


### The headline sentence

The number to quote in the report is:

> **For the baseline 0.25 m² detector and a 10 m radius chamber, the expected
> chamber signal reaches 5σ after approximately 13.4 days under the project's
> simplified model.**

The exact $\sqrt{T}$ value from the 1-day expectation is 13.33 days; the
interpolated scan gives 13.37 days. Quoting one decimal place, 13.4 days, is
honest. There is no uncertainty on this number because it is a deterministic
property of the model, not a measurement. The statistical spread of an actual
noisy observation is shown by the red points.

At 3σ the same curve crosses at about 4.8 days, which is the earlier waypoint
the handbook asks for.


## Why the noisy curve is different

If you repeat the analysis with Poisson observations, individual significance
values will jitter around the expected curve. That is not a bug. It is the
statistical fluctuation the simulation is designed to teach.

The previous version of this project used the maximum noisy pixel as the
headline statistic. We no longer do that: choosing a pixel because it fluctuated
upward introduces a selection bias. You may explore the maximum as a separate
exercise, but it is not the official exposure-time definition.


## Optional: the no-void control and the look-elsewhere effect

Run the same noisy significance calculation on the solid pyramid. This is a
**null simulation**, not a discovery test: there is no chamber, so every
positive pixel is noise. With 1,681 pixels in the image, a few 3σ pixels are
expected even when nothing is there.

The cell below runs 500 independent null experiments and records the largest
pixel in each. That distribution is the look-elsewhere effect in one figure.


In [7]:
rng_null = np.random.default_rng(cfg.REFERENCE_SEED + 1)
N_NULL = 500
null_max = np.empty(N_NULL)
null_n3 = np.empty(N_NULL, dtype=int)
null_n5 = np.empty(N_NULL, dtype=int)
null_at_signal = np.empty(N_NULL)
example_null = None

for k in range(N_NULL):
    obs = rng_null.poisson(expected_30d_0)
    S = mg.significance(obs, expected_30d_0)
    if k == 0:
        example_null = S.copy()
    null_max[k] = S.max()
    null_n3[k] = np.sum(S >= 3.0)
    null_n5[k] = np.sum(S >= 5.0)
    null_at_signal[k] = S[signal_idx]

print('null simulations                :', N_NULL)
print('mean/median max significance    :',
      f'{null_max.mean():.2f} / {np.median(null_max):.2f}')
print('95th percentile of max          :', f'{np.percentile(null_max, 95):.2f}')
print('fraction of null maps with max > 3 sigma:', f'{np.mean(null_max > 3):.3f}')
print('fraction of null maps with max > 5 sigma:', f'{np.mean(null_max > 5):.3f}')
print('mean number of pixels above 3 sigma     :', f'{null_n3.mean():.2f}')
print('mean number of pixels above 5 sigma     :', f'{null_n5.mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(example_null, origin='lower', extent=extent, aspect='auto',
                    cmap='RdBu_r', vmin=-5, vmax=6)
axes[0].set_xlabel('ax (degrees)')
axes[0].set_ylabel('ay (degrees)')
axes[0].set_title('Null simulation: solid pyramid, 30 days')
fig.colorbar(im, ax=axes[0], label=r'observed significance ($\sigma$)')

axes[1].hist(null_max, bins=30, color='C3', alpha=0.75, label='maximum pixel')
axes[1].hist(null_at_signal, bins=30, color='C0', alpha=0.55,
             label='one fixed pixel')
axes[1].axvline(3.0, color='C1', ls='--', lw=1)
axes[1].axvline(5.0, color='C2', ls='--', lw=1)
axes[1].set_xlabel(r'observed significance ($\sigma$)')
axes[1].set_ylabel('number of null experiments')
axes[1].set_title('Look-elsewhere: 500 noise-only experiments')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/12_null_ensemble.png', dpi=300, bbox_inches='tight')
plt.show()

null simulations                : 500
mean/median max significance    : 3.44 / 3.40
95th percentile of max          : 4.12
fraction of null maps with max > 3 sigma: 0.916
fraction of null maps with max > 5 sigma: 0.000
mean number of pixels above 3 sigma     : 2.68
mean number of pixels above 5 sigma     : 0.00


### Reading the null control

A single fixed pixel fluctuates around zero with a standard deviation of about
one, exactly as Poisson statistics predicts. The **maximum** over 1,681 pixels
does not: it has a median near 3.4σ and exceeds 3σ in more than nine out of ten
noise-only images. That is not a signal. It is the look-elsewhere effect.

The practical rule is the one the handbook states: a 3σ pixel in a wide image
is interesting, not decisive. The official exposure time therefore uses a
**pre-selected** signal pixel and the smooth expected-significance curve, not
the largest pixel in a noisy map. The maximum is retained here only as an
explicitly labelled exploratory statistic.


## How a real observation fluctuates around the curve

The red points on the exposure curve are the mean of 200 Poisson runs with an
error bar of one standard deviation. The cell below makes the underlying
distributions explicit for three exposures: 7 days (below 5σ), 14 days (near
5σ), and 30 days (well above).


In [8]:
rng_void = np.random.default_rng(cfg.REFERENCE_SEED + 2)
dist_days = [7.0, 14.0, 30.0]
distributions = {}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
for ax, d in zip(axes, dist_days):
    vals = []
    for _ in range(500):
        obs = rng_void.poisson(expected_1d * d)
        vals.append(mg.significance(obs, expected_1d_0 * d)[signal_idx])
    vals = np.array(vals)
    distributions[d] = vals
    ax.hist(vals, bins=30, color='C0', alpha=0.75)
    ax.axvline(5.0, color='C2', ls='--', lw=1.2, label=r'5$\sigma$')
    ax.axvline(S1_expected * np.sqrt(d), color='C3', ls=':', lw=1.2,
               label='expected')
    ax.set_title(f'{d:.0f} days')
    ax.set_xlabel(r'observed significance at fixed pixel ($\sigma$)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('number of experiments')
axes[-1].legend(frameon=False)
plt.tight_layout()
plt.savefig('../figures/13_observed_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

for d in dist_days:
    vals = distributions[d]
    print(f'{d:4.0f} days: expected {S1_expected*np.sqrt(d):6.3f}  '
          f'mean {vals.mean():6.3f}  s.d. {vals.std():5.3f}  '
          f'fraction > 5 sigma {np.mean(vals > 5):5.3f}')

   7 days: expected  3.624  mean  3.746  s.d. 1.194  fraction > 5 sigma 0.152
  14 days: expected  5.125  mean  5.116  s.d. 1.244  fraction > 5 sigma 0.540
  30 days: expected  7.502  mean  7.532  s.d. 1.226  fraction > 5 sigma 0.980


## A systematic check: the pixel solid angle

`mg.pixel_solid_angle` uses the small-angle approximation
$\Omega = (\Delta\theta)^2$. At the edge of the ±40° field this overestimates
the true solid angle by roughly 20–30 per cent, because a pixel in a
tangent-plane camera covers less solid angle as it tilts away from the normal.
The effect is smallest at the chamber (about 13 per cent at `ay = 30°`).

The official headline uses the supplied convention, as the project specifies.
The check below repeats the signal-pixel calculation with the exact
solid-angle factor for this camera parameterisation and shows how much the
answer moves. It is a systematic shift, not a statistical uncertainty.


In [9]:
def exact_pixel_solid_angle(ax_deg, ay_deg, step_deg):
    """Exact solid angle of a tangent-plane pixel, in steradians."""
    ax = np.radians(np.asarray(ax_deg, dtype=float))
    ay = np.radians(np.asarray(ay_deg, dtype=float))
    dax = np.radians(step_deg)
    day = np.radians(step_deg)
    tan2 = np.tan(ax)[None, :] ** 2 + np.tan(ay)[:, None] ** 2
    cos_theta = 1.0 / np.sqrt(1.0 + tan2)
    return ((1.0 / np.cos(ax)[None, :] ** 2) *
            (1.0 / np.cos(ay)[:, None] ** 2) *
            cos_theta ** 3 * dax * day)

used = mg.pixel_solid_angle(np.radians(cfg.STEP_DEG))
exact = exact_pixel_solid_angle(ax_deg, ay_deg, cfg.STEP_DEG)
factor = exact / used

S1_exact = mg.expected_significance(expected_1d * factor,
                                    expected_1d_0 * factor)[signal_idx]
S30_exact = S1_exact * np.sqrt(30.0)
t5_solid_angle = 30.0 * (5.0 / S30_exact) ** 2

print('solid-angle factor at signal pixel :', f'{factor[signal_idx]:.4f}')
print('solid-angle factor range in field  :',
      f'{factor.min():.3f} to {factor.max():.3f}')
print('S(30 d) with supplied approximation:', f'{mg.expected_significance(expected_30d, expected_30d_0)[signal_idx]:.4f}')
print('S(30 d) with exact pixel solid angle:', f'{S30_exact:.4f}')
print('5-sigma exposure, supplied approximation: 13.33 days')
print('5-sigma exposure, exact pixel solid angle:', f'{t5_solid_angle:.2f} days')

solid-angle factor at signal pixel : 0.8660
solid-angle factor range in field  : 0.766 to 1.000
S(30 d) with supplied approximation: 7.5016
S(30 d) with exact pixel solid angle: 6.9810
5-sigma exposure, supplied approximation: 13.33 days
5-sigma exposure, exact pixel solid angle: 15.39 days


## Summary of this notebook

- Poisson noise is real, and one day of data is almost pure noise.
- The **expected significance** is a smooth, deterministic curve; it reaches
  3σ at about 4.8 days and **5σ at about 13.4 days** for the baseline.
- An individual 30-day observation lands near 7.5σ but fluctuates by about
  ±1.2σ. At 14 days, roughly half of the experiments cross 5σ even though the
  expected value is 5.1σ.
- A noise-only image of a solid pyramid produces 3σ pixels in more than 90 per
  cent of experiments because there are 1,681 pixels. This is the
  look-elsewhere effect, and it is why the headline uses a fixed pixel and the
  expected curve.
- Using the exact pixel solid angle instead of the supplied small-angle
  approximation lengthens the 5σ exposure from 13.3 to about 15.4 days. That
  shift belongs in the discussion of systematics, not in the headline number.
